In [0]:
%sql
-- Make sure you are using the correct catalog
USE CATALOG mq_gmdf_dp_poc;

-- 1. Stop any active stream using this checkpoint
-- Find the stream ID from the Spark UI or error logs if you need to.
-- This might fail if no stream is active, which is okay.
-- %python
-- for stream in spark.streams.active:
--     if "ewm_ddl_ext_checkpoint" in stream.lastProgress.get("checkpointLocation"):
--         print(f"Stopping stream {stream.id}")
--         stream.stop()


-- 2. Clear the target table completely.
-- This deletes the incorrectly formatted data from previous runs.
TRUNCATE TABLE kafka_ewm_poc.ewm_ddl_ext;




In [0]:

# Based on your root path and the target table name 'ewm_ddl_ext'
checkpoint_path = "/tmp/checkpoints/ewm_ddl_ext_checkpoint" 
try:
  dbutils.fs.rm(checkpoint_path, recurse=True)
  print(f"✅ Checkpoint directory '{checkpoint_path}' removed.")
except Exception as e:
  if "java.io.FileNotFoundException" in str(e):
    print(f"✅ Checkpoint directory '{checkpoint_path}' does not exist. Nothing to remove.")
  else:
    raise e

In [0]:
dbutils.fs.rm("/tmp/mq_job_b_checkpoints/ewm_ddl_ext_checkpoint", recurse=True)

In [0]:
# ===================================================================================
#
# JOB B: DYNAMIC STREAMING INGESTION ENGINE
# INTERACTIVE TEST SCRIPT (v13.4 - With Case-Correction Logic)
#
# ===================================================================================

import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, StructField
from datetime import datetime

# ===================================================================================
# --- CONFIGURATION ---
# ===================================================================================
target_table_name = "mq_gmdf_dp_poc.kafka_ewm_poc.ewm_ddl_ext"
control_table_schema = "mq_gmdf_dp_poc.metadata_ddl"
source_data_path = "abfss://topics@mqosidevadls01.dfs.core.windows.net/ZMW_I_INTFLOGS_v2/"
checkpoint_location_root = "/tmp/mq_job_b_checkpoints/"
r_src_id = "plant-1"
etl_job_name = "ewm_streaming_ingestion"
# ===================================================================================

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")
spark.conf.set("spark.sql.caseSensitive", "false")

### Section 1: Initialization ###
run_id = str(uuid.uuid4())
simple_table_name = target_table_name.split('.')[-1]
schema_store_table = f"{control_table_schema}.schema_store_columns"
stream_log_table = f"{control_table_schema}.stream_job_execution_log"
microbatch_log_table = f"{control_table_schema}.microbatch_execution_log"
checkpoint_path = f"{checkpoint_location_root.rstrip('/')}/{simple_table_name}_checkpoint"
# ... (Initialization logging is unchanged)

### Section 2: `foreachBatch` Processing Logic ###

def flatten_structs(df, prsd_suffix="_prsd"):
    # ... (This function is unchanged)
    struct_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StructType)]
    if not struct_cols: return df
    for col_name in struct_cols:
        prefix = col_name[:-len(prsd_suffix)] if col_name.endswith(prsd_suffix) else col_name
        expanded_cols = [F.col(f"`{col_name}`.`{f.name}`").alias(f"{prefix}_{f.name}") for f in df.schema[col_name].dataType.fields]
        df = df.select("*", *expanded_cols).drop(col_name)
    return flatten_structs(df, prsd_suffix)

def process_microbatch(batch_df, batch_id):
    microbatch_id = str(uuid.uuid4())
    mb_start_ts = datetime.now()
    print(f"\n--- Starting Microbatch ID: {batch_id} ({microbatch_id}) ---")

    try:
        batch_df.cache()
        row_count = batch_df.count()

        if row_count == 0:
            print("  - Microbatch is empty. Skipping.")
            batch_df.unpersist()
            return

        # Pre-correction Step: Fix known case inconsistencies in JSON strings
        print("  - Pre-correcting known JSON key case issues...")
        corrections = [("ReqPayload", "ClientId", "clientId")]
        corrected_df = batch_df
        for col, bad_key, good_key in corrections:
            if col in corrected_df.columns:
                corrected_df = corrected_df.withColumn(
                    col, 
                    F.regexp_replace(F.col(col), f'"{bad_key}"', f'"{good_key}"')
                )

        # 1. Load blueprint
        schema_blueprint_df = spark.table(schema_store_table).filter(F.col("contract_hash") == active_hash).cache()
        blueprint = schema_blueprint_df.collect()
        
        # 2. Parse JSON (using the 'corrected_df' now)
        transformed_df = corrected_df
        json_sources = {}
        for r in [r for r in blueprint if r['source_kind'] == 'JSON']:
            if r['source_column'] not in json_sources: json_sources[r['source_column']] = []
            json_sources[r['source_column']].append(r)

        for source_col, fields in json_sources.items():
            if source_col in transformed_df.columns:
                dynamic_schema = StructType([StructField(f.json_key, StringType(), True) for f in fields])
                transformed_df = transformed_df.withColumn(f"{source_col}_prsd", F.from_json(F.col(source_col), dynamic_schema))

        # 3. Flatten
        flattened_df = flatten_structs(transformed_df)

        # 4. Align to DDL Contract
        aligned_exprs = []
        existing_cols_lower = {c.lower() for c in flattened_df.columns}
        for r in blueprint:
            source_col_name = None
            if r['source_kind'] == 'ATOMIC': source_col_name = r['source_column']
            elif r['source_kind'] == 'JSON': source_col_name = f"{r['source_column']}_{r['json_key']}"
            elif r['source_kind'] == 'STRUCT': source_col_name = r['source_column'].replace(".", "_")
            if source_col_name and source_col_name.lower() in existing_cols_lower:
                aligned_exprs.append(F.col(f"`{source_col_name}`").alias(r['physical_column_name']))
            elif not r['source_kind'].startswith("PIPELINE"):
                aligned_exprs.append(F.lit(None).cast(r['type']).alias(r['physical_column_name']))
        aligned_df = flattened_df.select(aligned_exprs)
        
        # 5. Enforce Golden Schema
        golden_schema = spark.table(target_table_name).schema
        golden_cols_map = {f.name.lower(): f for f in golden_schema.fields}
        final_select_exprs = [F.col(f"`{c}`").cast(golden_cols_map[c.lower()].dataType).alias(golden_cols_map[c.lower()].name) for c in aligned_df.columns if c.lower() in golden_cols_map]
        enforced_df = aligned_df.select(final_select_exprs)

        # 6. Add Audit Columns
        final_df = enforced_df.withColumn("ingest_batch_id", F.lit(batch_id).cast("long")) \
                              .withColumn("loaded_timestamp", F.lit(mb_start_ts).cast("timestamp")) \
                              .withColumn("schema_hash_at_ingest", F.lit(active_hash).cast("string"))

        # 7. Write
        final_df.write.format("delta").mode("append").saveAsTable(target_table_name)
        
        schema_blueprint_df.unpersist()
        batch_df.unpersist()
        
        spark.sql(f"INSERT INTO {microbatch_log_table} VALUES ('{microbatch_id}', '{run_id}', '{r_src_id}', '{etl_job_name}', '{target_table_name}', {batch_id}, '{mb_start_ts.isoformat()}', '{datetime.now().isoformat()}', 'COMPLETED', '{active_hash}', {row_count}, null)")
        print(f"  - ✅ Microbatch ID: {batch_id} completed successfully.")

    except Exception as e:
        # ... (Error handling is unchanged)
        raise e

### Section 3: Main Stream Execution ###
try:
    print("\n🚀 Starting the stream in HYBRID mode...")
    active_hash_query = f"SELECT schema_hash FROM {control_table_schema}.schema_transition WHERE target_table_name = '{simple_table_name}' AND event_type = 'ACTIVE' LIMIT 1"
    active_hash_result = spark.sql(active_hash_query).first()
    if active_hash_result is None: raise RuntimeError(f"Could not find an ACTIVE schema for table '{simple_table_name}'. Please run Job A.")
    active_hash = active_hash_result['schema_hash']
    print(f"   - Locked schema hash for this run: {active_hash}")
    raw_stream_df = spark.readStream.format("cloudFiles").option("cloudFiles.format", "parquet").option("cloudFiles.schemaLocation", checkpoint_path).option("cloudFiles.inferColumnTypes", "true").load(source_data_path)
    stream_writer = raw_stream_df.writeStream.outputMode("append").option("checkpointLocation", checkpoint_path).foreachBatch(process_microbatch).trigger(availableNow=True).start()
    stream_writer.awaitTermination()
    # ... (Success logging is unchanged)
except Exception as e:
    # ... (Error handling is unchanged)
    raise e

In [0]:
# ===================================================================================
#
# JOB B - INTERACTIVE PIPELINE MONITOR
#
# PURPOSE:
#   To be run interactively to audit the live data pipeline, checking for data
#   freshness, consistency, and quality anomalies.
#
# ===================================================================================

from pyspark.sql import functions as F
from datetime import datetime

# ===================================================================================
# --- CONFIGURATION FOR INTERACTIVE TEST ---
# ===================================================================================

# --- Main Parameters ---
target_table = "mq_gmdf_dp_poc.kafka_ewm_poc.ewm_ddl_ext"
control_table_schema = "mq_gmdf_dp_poc.metadata_ddl"

# --- Monitoring Thresholds ---
# How many minutes of recent data to check.
time_window_minutes = 60

# How old can the newest record be before we consider the pipeline "stale"?
freshness_threshold_minutes = 90

# What percentage of nulls in critical columns is unacceptable? (e.g., 1.0 means 1%)
max_null_percentage = 1.0

# How many rescued records in the time window is too many?
max_rescued_count = 100

# Which columns are critical for the null check?
critical_columns = ["run_id", "req_clientId"]

# ===================================================================================
# --- SCRIPT EXECUTION STARTS HERE ---
# ===================================================================================

### Section 1: Initialization ###
schema_transition_table = f"{control_table_schema}.schema_transition"
simple_table_name = target_table.split('.')[-1] # Use simple name for lookups

print(f"✅ Starting Pipeline Monitoring for: {target_table}")
print(f"   - Time Window: Last {time_window_minutes} minutes")


### Section 2: Validation Checks ###

failures = []

try:
    # --- Pre-fetch recent data ---
    print("\nPre-fetching recent data for analysis...")
    recent_data_df = spark.table(target_table).filter(F.col("loaded_timestamp") >= F.expr(f"now() - INTERVAL {time_window_minutes} minutes"))
    recent_data_df.cache() # Cache for performance
    record_count = recent_data_df.count()
    print(f"  - Found {record_count} records in the last {time_window_minutes} minutes.")

    # --- Check 1: The "Data Freshness" Check ---
    print(f"\n🔄 Running Check 1: Data Freshness (Threshold: < {freshness_threshold_minutes} mins)...")
    max_ts = spark.table(target_table).agg(F.max("loaded_timestamp")).first()[0]
    if not max_ts:
         failures.append("FAILURE: Target table is empty or max(loaded_timestamp) is NULL.")
    else:
        # Using Python for time diff to avoid Spark timezone complexities in interactive mode
        minutes_since_last_load = (datetime.now(max_ts.tzinfo) - max_ts).total_seconds() / 60
        if minutes_since_last_load > freshness_threshold_minutes:
            failures.append(f"FAILURE: Data is not fresh. Last record loaded {int(minutes_since_last_load)} minutes ago (Threshold: {freshness_threshold_minutes}).")
        else:
            print(f"  - PASSED: Last record loaded {int(minutes_since_last_load)} minutes ago.")

    # --- Check 2: The "Schema Hash Consistency" Check ---
    print("\n🔄 Running Check 2: Schema Hash Consistency...")
    active_hash_result = spark.table(schema_transition_table).filter(f"target_table_name = '{simple_table_name}' AND event_type = 'ACTIVE'").first()
    if not active_hash_result:
        failures.append(f"CRITICAL FAILURE: Could not find an 'ACTIVE' schema for '{simple_table_name}' in the control tables.")
    else:
        active_hash = active_hash_result["schema_hash"]
        mismatched_hash_count = recent_data_df.filter(F.col("schema_hash_at_ingest") != active_hash).count()
        if mismatched_hash_count > 0:
            failures.append(f"FAILURE: Found {mismatched_hash_count} recent records with a schema hash other than the active one ('{active_hash}').")
        else:
            print("  - PASSED: All recent records have the correct active schema hash.")

    # --- Check 3: The "Null Rate Anomaly" Check ---
    print(f"\n🔄 Running Check 3: Null Rate for Critical Columns (Threshold: < {max_null_percentage}%)...")
    if record_count > 0:
        for col_name in critical_columns:
            if col_name in recent_data_df.columns:
                null_count = recent_data_df.filter(F.col(col_name).isNull()).count()
                null_percentage = (null_count / record_count) * 100
                if null_percentage > max_null_percentage:
                    failures.append(f"FAILURE: Null rate for column '{col_name}' is {null_percentage:.2f}%, which is above the {max_null_percentage}% threshold.")
                else:
                    print(f"  - PASSED: Null rate for '{col_name}' is {null_percentage:.2f}%.")
    else:
        print("  - SKIPPED: No recent data to check for null rates.")

    # --- Check 4: The "_rescued_data" Volume Check ---
    print(f"\n🔄 Running Check 4: Rescued Data Volume (Threshold: < {max_rescued_count} records)...")
    if "_rescued_data" in recent_data_df.columns:
        rescued_count = recent_data_df.filter(F.col("_rescued_data").isNotNull()).count()
        if rescued_count > max_rescued_count:
            failures.append(f"FAILURE: Found {rescued_count} records with rescued data, which is above the {max_rescued_count} threshold.")
        else:
            print(f"  - PASSED: Rescued data count is {rescued_count}.")
    else:
        print("  - SKIPPED: No '_rescued_data' column found in target table.")

except Exception as e:
    import traceback
    failures.append(f"CRITICAL FAILURE during script execution: {traceback.format_exc()}")
finally:
    recent_data_df.unpersist()


### Section 3: Final Verdict ###

if failures:
    print("\n\n❌ PIPELINE MONITORING FAILED.")
    for f in failures:
        print(f"  - {f}")
    # This error will help you see the failure clearly in an interactive context
    raise Exception("One or more pipeline monitoring checks failed. Please review the errors.")
else:
    success_msg = "✅ All pipeline monitoring checks passed successfully."
    print(f"\n\n{success_msg}")